<h1 style="font-size:50px; font-family:Monaco; text-align:center">IntroSec - PWN, or Binary Exploitation</h1>
<h1 style="font-size:30px; font-family:Monaco; text-align:center">Part 5: ASLR and PIE
</h1>

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">1 - Third Defense</h1>

Ready for more bad news? Stack cookies are a simple defense and for better or worse, are also simple to bypass. NX restricts our payloads quite a bit but is still workable. Something that is quite a bit more complicated (and much more difficult to defeat) is Address Space Layout Randomization (`ASLR`) alongside Position Independent Executables (`PIE`, also known as Position Independent Code or `PIC`).

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.1 - What is ASLR?</h2>

`ASLR` is a kernel-level mitigation in basically every modern operating system that randomizes the memory layout of programs that are executed. What this means more specifically is that instead of using fixed addresses for things like the stack, heap, and `libc`, the items in those locations are each given an offset which is applied to a randomly selected base address.

If you wanted to disable `ASLR` on your system (for some reason), it can be done with the following command (the cell doesn't execute anything on your system so don't worry :)):

```c
echo 0 | sudo tee /proc/sys/kernel/randomize_va_space
```

It can be re-enabled very simply by running the above command with a 2 instead.

<em>From a <a href="https://oneuptime.com/blog/post/2026-03-02-how-to-enable-and-configure-address-space-layout-randomization-aslr-on-ubuntu/view">blog post</a> about ASLR config on Linux:</em>

<img src="./media/kernelExplanation.png">

ONE LAST IMPORTANT NOTE!!!

By default, GDB will disable `ASLR` for the binary you are running. It is important to use either `checksec` to verify a binary's defenses, or alternatively, you can run the below command in GDB BEFORE running your file to disable `ASLR`:

```c
set disable-randomization off
```

Bear in mind that the command's effects only last for that current session. Let's take a look at an example file in GDB.

In [1]:
!mkdir -p ~/introsec/pwn/aslr/binaries

In [2]:
%cd ~/introsec/pwn/aslr/binaries

/home/user/introsec/pwn/aslr/binaries


Here is our very simple example function:

In [3]:
%%file example.c

#include <stdlib.h>

// gcc example.c -o example

int main(void){
    return 0;http://localhost:8888/files/pwn/aslr/media/withoutASLR.png?_xsrf=2%7C344c7466%7C6fc16c57b532cf0225236ca6a0b02b16%7C1786291033&1788035679264
}

Writing example.c


In [4]:
!gcc example.c -o example

<em>Output of `vmmap` in GDB without `ASLR` enabled</em>

<img src="./media/withoutASLR.png">

<em>Output of `vmmap` in GDB with `ASLR` enabled</em>

<img src="./media/withASLR.png">

Notice how different the two are? `ASLR` is very powerful (and annoying), but a file doesn't <em>have</em> to use it. Let's talk about `PIE`!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.2 - What is a PIE?</h2>

A pie is a dessert consisting of a filling (as of fruit or custard) in a pastry shell or topped with pastry or both. `PIE` is a compilation mechanism that allows for `ASLR` to be utilized for the user-written code in a binary (e.g. for stuff that isn't on the heap, stack, or in the C library).

What this means is that the top section of the memory map (the portion that is mapped from the file you are running) can now be randomized. If you go back and look at the pictures above, you can see this working quite nicely. Let's take a look at two examples using `objdump`, one without `PIE` and one with.

<em>`objdump` of file compiled without `PIE`</em>

<img src="./media/objdumpWithoutPIE.png">

<em>`objdump` of file compiled with `PIE`</em>

<img src="./media/objdumpWithPIE.png">

As we can see in the above two images, the file compiled without `PIE` has absolute addresses. Those addresses are the same ones that will be loaded into memory i.e. if you disassemble 1 instruction at address `0x40110a`, you will always see `push rbp`.

In the second image, however, we have much smaller numbers. What these represent are the offsets from the (as of now) unknown base address that will be used for the binary. When it is loaded into memory and a base address is selected, adding `0x1129` to the base address will take you to the start of `main`.

Now that we have seen them at rest, let's see what changes in GDB.

In [5]:
!gcc example.c -no-pie -o withoutPIE

<em>File compiled without `PIE` and with `ASLR` enabled</em>

<img src="./media/withoutPIEwithASLR.png">

In [6]:
!gcc example.c -o withPIE

<em>File compiled with `PIE` and with `ASLR` enabled</em>

<img src="./media/withPIEwithASLR.png">

As we can see, the portion of memory dedicated to the binary itself is only randomized when `PIE` AND `ASLR` are enabled.

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">1.3 - What does this mean?</h2>

We now need to add another step to the process of exploiting a binary. Should `ASLR`/`PIE` be enabled, if we want to jump anywhere (whether it be to a ROP chain, a win function, shellcode, etc.), we need to calculate the base address of that region of memory.

The base addresses of the binary, libc, the stack, and the heap will all be different each time (and at different distances from each other). Bear in mind, however, that the offsets within these regions are consistent across runs. That is, while the base address of libc will be different each run, the offset from the base address to `system` will be the same each time.

Therefore, if we can figure out the base address of libc while the binary is running (and get control of `RIP`), we can simply add the offset for `system` to the base address of libc and get there!

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">2 - Third Impact</h1>

So, how do we accomplish this?

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.1 Still leaking</h2>

It is precisely the same as stack cookies! You are simply looking to leak an address in the region you desire (e.g. if you want to leak libc's base address, leak the address of `fgets()`). The actual way you leak an address is, again, quite binary-specific. For example, maybe you can overwrite a buffer in a struct that sits next to a function pointer and then print it out.

This is where the creativity of hacking comes in! All you need to do is get the program to print out an address!

Also remember, it doesn't have to be a function! Functions are convenient, yes, because you can easily find their offset from the base address. However, if it is an address that is within that range, it will have a constant offset to the base address (though it might be a little more troublesome to find it).

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.2 Offset calculation</h2>

Now that we have an address, what do we do with it?

For starters, we need to know the offset from the base address of the region of memory your address lives in. This can be accomplished in a few ways, but here is the simplest (not necessarily the fastest):

IMPORTANT NOTE: All below steps must be done on the same run. Re-running the binary will cause the addresses to change!

1. Figure out what function you are actually leaking (often times this can be sussed out by looking at memory layouts, if not, you can disassemble at that address to see what function is being called). Let's assume we leaked the address of `printf`
2. Determine that function's address in GDB with a command such as `p printf`
3. Find `libc`'s base address using `vmmap`
4. Subtract `libc`'s base address from your function address (in this example, from `printf`) with a command like `p/x <printf_address>-<libc_address>`
5. Now you have your offset!

But we aren't quite done! In order to make this usable for jumping to other functions or finding gadgets within `libc`, we need to use the base address. The way we get the base address during an actual run is simple: since we already know the offset of the function we leaked (remember, this doesn't change between runs, only the base address does), we can read the leak in with `pwntools` and subtract the offset from it to get the base address!

<hr style="border:1px solid white">

<h2 style="font-size:20px; font-family:Monaco">2.3 Base (address) of operations</h2>

Now that we have the base address, we are off to the races! From here, it is much the same as before `ASLR`/`PIE` was a thing.

If you want to jump to a function, you just need to add its offset to the base address.

If you want to use gadgets, you just need to add their offsets to the base address!

<hr style="border:1px solid white">

<h1 style="font-size:30px; font-family:Monaco">Practice:</h1>

Leak into ROP